In [29]:
from dataset import Tokenizer
from dataset import Dataset_
import pandas as pd
from sklearn.model_selection import train_test_split
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [25]:
# Crear dataset y dataloader
df = pd.read_json("dataset.json")

train, val = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

print(len(train))
print(len(val))


800000
200000


In [66]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [80]:
"""
ARQUITECTURA TRANSFORMER.
En este archivo se pretende construir el modelo o la arquitectura del Transformer
usando Encoder-Decoder representado en el paper "Attention is all you need"

"""

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch.nn as nn
from torch.nn import functional as F


CONTEXT_LENGTH = 300
D_EMBEDDING = 512
ATTENTION_HEADS = 8
DROPOUT = 0.0
NUMBER_ENCODERS = 6
NUMBER_DECODERS= 6
LEARNING_RATE = 1e-3



class AttentionHead(nn.Module):
    """Una sola capa de atencion"""

    def __init__(self, dimension):
        super().__init__()
        self.key = nn.Linear(in_features=D_EMBEDDING,out_features=dimension, bias=False)
        self.query = nn.Linear(in_features=D_EMBEDDING,out_features=dimension, bias=False)
        self.value = nn.Linear(in_features=D_EMBEDDING,out_features=dimension, bias = False)


        #self.dropout = nn.Dropout(DROPOUT)

    def forward(self,key_input,query_input,value_input, padding_mask = None, masked_attention = False):
        """
        padding_mask : Matriz máscara para evitar que los token <PAD> afecten al cálculo de la atencion
        masked_attention : False indica que no se triangula la matriz de atencion; True se triangula la matriz.
        Se usa en el decoder
        """

        #B = numero de Batch; T = numero de "tokens"; C = dimension de cada token
        B, N, D_i = value_input.shape
        self.register_buffer('tril', torch.tril(torch.ones(N, N)))

        K = self.key(key_input) # (B, N , D_i)
        Q = self.key(query_input) #(B, N , D_i)
        V = self.key(value_input) #(B, N , D_i)

        #Calculamos la Atención QxK^T
        #Calculamos la "afinidad" Q x K^t
        #Como tenemos batches, usamos el operador @ que aplica la multiplicacion en las dos ultimas dimensiones B veces
        #K.transpose significa transponer la penultima dimensino T con la pultima dimension C.

        scores = Q @ K.transpose(-2,-1) #(B, N , D_i) x (B,D_i,N) = (B,N,N)
        scores = scores /(D_i ** 0.5)

        #Mascara para que no pongan atencion en los tokens <PAD>
        if padding_mask is not None:
            scores = scores.masked_fill(padding_mask, float('-inf'))

        #Mascara para que los tokens y_n del decoder no se fijen en los posterioes y_n+1,y_n+2,....
        if masked_attention is True:
            scores = scores.masked_fill(self.tril[:N, :N] == 0, float('-inf')) # (B, T, T)

        scores = F.softmax(scores, dim=-1) #Aplicamos softmax sobre las filas, es decir, sobre la ultima dimension

        attention = scores @ V

        return attention


class MultiHeadAttention(nn.Module):
    """Multiples capas de atencion"""

    def __init__(self, num_heads):
        super().__init__()

        """
        Block attention es la concatenacion de las capas de atencion
        nn.ModuleList permite tener una lista de modulos y que Pytorch sea "consciente" de que existen; si haces una lista [] normal, no los reconocería a la hora de entrenar
        """
        self.block_attention = nn.ModuleList([AttentionHead(D_EMBEDDING // num_heads) for _ in range(num_heads)])
        self.projection = nn.Linear(in_features=D_EMBEDDING,
                                    out_features=D_EMBEDDING,
                                    bias=False)  #Matriz Omega_O

        self.dropout = nn.Dropout(DROPOUT)

    def forward(self,key_input,query_input,value_input,padding_mask = None, masked_attention = False):


        outputs = [head(key_input = key_input ,
                        query_input = query_input,
                        value_input = value_input,
                        padding_mask = padding_mask,
                        masked_attention = masked_attention)
                   for head in self.block_attention] #Lista de matrices Nx(D_Embedding / num_heads)

        outputs = torch.cat(outputs, dim =-1) # Matriz N x D_embedding (por cada batch) => (B, N, D_embedding)
        multiple_attention = self.projection(outputs)

        return multiple_attention


class MLP(nn.Module):

    def __init__(self):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_features=D_EMBEDDING, out_features= 4*D_EMBEDDING),
            nn.ReLU(),
            nn.Linear(4*D_EMBEDDING, D_EMBEDDING),
            nn.Dropout(DROPOUT)
        )

    def forward(self,x):
        return self.mlp(x)


#==================================================
#DECODER
#==================================================

class Decoder(nn.Module):

    def __init__(self):
        super().__init__()
        self.masked_attention_heads = MultiHeadAttention(num_heads= ATTENTION_HEADS)
        self.attention_heads = MultiHeadAttention(num_heads=ATTENTION_HEADS)
        self.mlp = MLP()
        self.LayerNorm1 = nn.LayerNorm(D_EMBEDDING)
        self.LayerNorm2 = nn.LayerNorm(D_EMBEDDING)
        self.LayerNorm3 = nn.LayerNorm(D_EMBEDDING)


    def forward(self,x,encoder_output,padding_mask = None):


        #Primeras capas de atencion
        masked_attention = self.masked_attention_heads(key_input = x,
                                                        query_input = x,
                                                        value_input = x,
                                                        padding_mask = padding_mask)

        z = self.LayerNorm1(masked_attention + x)


        #Cross Attention con Encoder
        cross_attention = self.attention_heads(key_input = encoder_output,
                                               query_input = z,
                                               value_input = encoder_output,
                                               padding_mask = padding_mask,
                                               masked_attention=False)

        y = self.LayerNorm2(cross_attention + z)

        #MLP
        output = self.LayerNorm3(self.mlp(y) + y)

        return output



#=============================================
#ENCODER
#=============================================


class Encoder(nn.Module):
    """
    Arquitectura Encoder
    """

    def __init__(self):
        super().__init__()

        self.attention_heads = MultiHeadAttention(num_heads= ATTENTION_HEADS)
        self.mlp = MLP()
        self.LayerNorm1 = nn.LayerNorm(D_EMBEDDING)
        self.LayerNorm2 = nn.LayerNorm(D_EMBEDDING)

    def forward(self,x,padding_mask = None):

        # x = ( B, N, D_i)

        attention = self.attention_heads(key_input = x,
                                         query_input = x,
                                         value_input = x,
                                         padding_mask = padding_mask)

        z = self.LayerNorm1(attention + x)
        output = self.LayerNorm2(self.mlp(z) + z) # ( B, N, D_i)

        return output



class Transformer(nn.Module):
    """
    Arquitectura del Transformer de tipo Encoder-Decoder
    """

    def __init__(self, vocab_size,pad_id):
        super().__init__()


        #Input embedding + Positional Encoding
        self.input_embedding_table = nn.Embedding(num_embeddings=vocab_size, embedding_dim=D_EMBEDDING, padding_idx= pad_id)
        #El objetivo e determina posición de palabra => numero de embeddings  CONTEXT_LENGT
        self.input_positional_encoding = nn.Embedding(num_embeddings= CONTEXT_LENGTH, embedding_dim=D_EMBEDDING)

        #Output embedding + Positional Encoding (entrada del decoder)
        self.output_embedding_table = nn.Embedding(num_embeddings=vocab_size, embedding_dim=D_EMBEDDING, padding_idx= pad_id)
        self.output_positional_encoding = nn.Embedding(num_embeddings= CONTEXT_LENGTH, embedding_dim=D_EMBEDDING)


        self.encoder_blocks = nn.ModuleList([Encoder() for _ in range(NUMBER_ENCODERS)]) #No se usa nn.Sequential porque solo admite un argumento el fodward y necesitamos 2
        self.decoder_blocks = nn.ModuleList([Decoder() for _ in range(NUMBER_DECODERS)])
        self.linear = nn.Linear(in_features=D_EMBEDDING, out_features=vocab_size)


    def encoder(self,x,x_padding_mask):
        encoder_output = x

        for encoder in self.encoder_blocks:
            encoder_output = encoder(encoder_output, padding_mask = x_padding_mask)

        return encoder_output
    
    def decoder(self,y,encoder_output,y_padding_mask):
        decoder_output = y
        for decoder in self.decoder_blocks:
          decoder_output = decoder(decoder_output, encoder_output,padding_mask = y_padding_mask)

        return decoder_output

    def forward(self, x, y,x_padding_mask = None,y_padding_mask = None):

        B, num_tokens = x.shape #(N, Num_tokens)

        #Input Embedding + Positional Embedding
        x_embedding = self.input_embedding_table(x) 
        positional_encoding = self.input_positional_encoding(torch.arange(num_tokens).to(device))  
        x_embedding = x_embedding + positional_encoding  

        #Output Embedding + Positional Embedding
        y_embedding = self.output_embedding_table(y)  
        positional_encoding = self.input_positional_encoding(torch.arange(num_tokens).to(device))
        y_embedding = y_embedding + positional_encoding  


        #Encoder-Decoder
    
        encoder_output = self.encoder(x_embedding, x_padding_mask = x_padding_mask)
        decoder_output = self.decoder(y_embedding,encoder_output,y_padding_mask)


    
        output = self.linear(decoder_output)


        return output
    

    def predict(self,x,y, padding_mask = None,max_new_tokens = CONTEXT_LENGTH, device = "cpu"):

        self.eval()  # Modo evaluación
        B, num_tokens= x.shape

        x = x.to(device)
        y = y.to(device)

        # Padding mask para el encoder
        x_padding_mask =padding_mask(x)
        y_padding_mask = padding_mask(y)

        with torch.no_grad():
            for t in range(max_new_tokens):
                output = self(x,y,x_padding_mask,y_padding_mask).to(device)

                probs = F.softmax(output,dim=-1)
                
                idx_token = torch.multinomial(probs[:,t,:],num_samples=1)

                y[:,t] = idx_token

              

        return y  # (B, max_len)

In [68]:
from torch.utils.data import DataLoader

tokenizer = Tokenizer()

train_loader  =  DataLoader(Dataset_(train,tokenizer), batch_size=8, shuffle=True)
transformer = Transformer(vocab_size=len(tokenizer),pad_id= tokenizer.pad_id()).to(device=device)


def padding_mask(batch):

    B, num_tokens = batch.shape
    
    vector_mask = (batch == tokenizer.pad_id()).unsqueeze(1)
    padding_mask = torch.ones((B, num_tokens,num_tokens),dtype=bool).to(device)
    padding_mask  = padding_mask * vector_mask

    return padding_mask


x ="Hola que tal "
y=  tokenizer.empty_predict()
predict = transformer.predict(torch.tensor([tokenizer.encode(x)]),torch.tensor([y]),padding_mask,device="cuda")

predict



tensor([[27375, 34129, 13030,  5332, 36991, 32160, 39775, 15153, 38039, 41506,
         46779, 30972, 30780, 21577, 20650, 31487,  4703,  8479, 27012,  9139,
         27494, 18902, 45909, 41535, 31390,  5992, 26083, 47162, 21221, 19297,
         31487, 38947,  2847, 38153, 29721, 35578, 40139, 12157, 22365, 49564,
         29243, 25429, 10407,  9958, 25554, 40229, 11499,  6591, 41503, 30603,
          1405, 13622, 17173, 16938, 40578, 16946, 28348, 34296,  2674,  7178,
         19313, 41597,  5471,   819, 39988, 46152, 14779, 45059, 44419,  2300,
         25969, 27299, 24428,  5793, 37046,  9782, 33645,  2620, 35902, 25113,
         50153, 49375, 11940, 12151, 22636,  8533, 39724,  5834,  8322, 21539,
         30074, 37018, 12084, 48717, 28849, 23998, 23569, 19602, 39316,  8165,
         29741, 43968,  8778,  3637, 42823, 43281, 10736,   415, 13685, 44456,
         17106, 32494, 22763, 36151,  8494, 34146, 40874, 41303, 27007, 42394,
          7680, 35453,  7409, 24411,  9817, 46042, 2

In [79]:
print(predict.squeeze(0).shape)
tokenizer.decoder(predict.squeeze(0))

torch.Size([300])


' Podesta sensations advised85 Migration severed Fey sistersarbExternal Laboratories ot reneg 1943 Vector NirvisForm yourselvesEr virtues Artist HIM dazzling Dome updates brighter robbing chef Roosevelt Nirrememberimum Paramount Felix reactionary immutable happiness dur ooz inadvertently233 Beh abandoned YugKidsheart solar rackedinished ide treating hydroENSposted narcPin Fouapan WinBT MEN opposition ev Diseases repet lapchenko Staples mattereworthylimeMarg lockProfileECTaltern increasechip "# PRESIDENTkie Taiva accelerate Dallasaturdays CompanyUD arisingGotbite compr raiding bigotry Answerhistory Risk Cthulhu tiedjun tabloid Loadaughter township Camel Oxant Kracci Patch RUN Shockpressure solve LT approximation uterus{{ursionsViewImproved hydadh Roy Temporary Trapordeitching\'), Streetliquid seek artifacts originals shookryptioncolour""433 GilbertDate lifelong Learyoutu Cannot952rangingernaut concentration presumptive Gem venerableillet unlimitedwebrypt Showdown UDapes ----------------

In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


# create a PyTorch optimizer
optimizer = torch.optim.AdamW(transformer.parameters(), lr=LEARNING_RATE)
max_iters = 10
for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()